# Chapter 9: Cryptography

Sending a secret message requires two things: a way to scramble it and a way
to unscramble it. Classical cryptography has been doing this for millennia.
Quantum mechanics adds something new: the laws of physics themselves can
guarantee that nobody is listening.

This notebook covers classical encryption (Caesar cipher, One-Time-Pad),
three quantum key exchange protocols (BB84, B92, EPR), and quantum
teleportation. Each section includes working Python code.

In [1]:
import numpy as np
from numpy.linalg import norm
import random
from math import sqrt

## 9.1 Classical Cryptography

**Cryptography** is the art of concealing messages. The word comes from
Greek: *krypton* (hidden) and *graphein* (writing).

The basic setup: Alice wants to send a message to Bob over an insecure
channel. She uses an encryption function $ENC$ with an encryption key
$K_E$ to transform her plaintext $T$ into ciphertext $E$:

$$ENC(T, K_E) = E \tag{9.1}$$

Bob applies a decryption function $DEC$ with decryption key $K_D$:

$$DEC(E, K_D) = T \tag{9.2}$$

The fundamental requirement: for every message $T$,

$$DEC(ENC(T, K_E), K_D) = T \tag{9.3}$$

Use the right keys, get the original message back. No information lost.

### Caesar's Protocol

Arrange the 26 English letters on a circle. The encryption key is an
integer $n$: shift each letter $n$ positions clockwise. Decryption shifts
back by $n$ (so $K_D = -K_E$).

$$ENC = DEC = \text{shift}(-, -)$$
$$\text{shift}(T, n) = T' \tag{9.4}$$

For example, shift("MOM", 3) = "PRP".

#### Programming Drill 9.1.1
*Implement the encryption and decryption of text with Caesar's protocol.
Using ASCII makes this particularly easy.*

In [2]:
def caesar_encrypt(text, shift):
    """Encrypt text using Caesar cipher with given shift."""
    result = []
    for ch in text.upper():
        if 'A' <= ch <= 'Z':
            result.append(chr((ord(ch) - ord('A') + shift) % 26 + ord('A')))
        else:
            result.append(ch)
    return ''.join(result)


def caesar_decrypt(text, shift):
    """Decrypt by shifting in the opposite direction."""
    return caesar_encrypt(text, -shift)


# Demonstrate
plaintext = "HELLO WORLD"
shift = 3
encrypted = caesar_encrypt(plaintext, shift)
decrypted = caesar_decrypt(encrypted, shift)

print(f"Plaintext:  {plaintext}")
print(f"Encrypted:  {encrypted}")
print(f"Decrypted:  {decrypted}")
assert decrypted == plaintext

Plaintext:  HELLO WORLD
Encrypted:  KHOOR ZRUOG
Decrypted:  HELLO WORLD


#### Exercise 9.1.2
*Decipher the message "JNTGMNF VKRIMHZKTIAR BL YNG." (Hint: Use Figure 9.2.)*

We try all 26 shifts to find the one producing readable English.

In [3]:
ciphertext = "JNTGMNF VKRIMHZKTIAR BL YNG"
# Try all shifts (brute force the 26-letter keyspace)
for s in range(26):
    decoded = caesar_decrypt(ciphertext, s)
    if "QUANTUM" in decoded or "CRYPTO" in decoded:
        print(f"Shift {s}: {decoded}")
        break
else:
    # Show all to find it manually
    for s in range(26):
        print(f"Shift {s:2d}: {caesar_decrypt(ciphertext, s)}")

Shift 19: QUANTUM CRYPTOGRAPHY IS FUN


The answer (shift 5): **QUANTUM CRYPTOGRAPHY IS FUN**.

The Caesar cipher is easy to break. Eve can try all 26 shifts in seconds,
and frequency analysis on longer messages reveals the key even faster.
The encrypted and original texts are *highly correlated*: the same
statistical patterns appear in both.

### One-Time-Pad (Vernam Cipher)

To eliminate statistical correlation, Alice and Bob use a random key $K$
the same length as the message. Both encryption and decryption are
bitwise XOR ($\oplus$):

$$K_E = K_D = K \tag{9.5}$$
$$ENC(T, K) = DEC(T, K) = T \oplus K \tag{9.6}$$

This works because XOR is its own inverse:

$$DEC(ENC(T, K), K) = (T \oplus K) \oplus K = T \oplus (K \oplus K) = T \tag{9.7}$$

The One-Time-Pad is provably secure: if $K$ is truly random and used
only once, the ciphertext $E$ carries zero information about $T$. Every
possible plaintext is equally likely given $E$.

#### Programming Drill 9.1.2
*Implement the One-Time-Pad protocol.*

In [4]:
def generate_key(length):
    """Generate a random binary key of given length."""
    return [random.randint(0, 1) for _ in range(length)]


def otp_encrypt(plaintext_bits, key):
    """Encrypt using XOR with key."""
    return [t ^ k for t, k in zip(plaintext_bits, key)]


def otp_decrypt(ciphertext_bits, key):
    """Decrypt using XOR with key (same operation)."""
    return [e ^ k for e, k in zip(ciphertext_bits, key)]


def text_to_bits(text):
    """Convert ASCII text to list of bits."""
    bits = []
    for ch in text:
        for i in range(7, -1, -1):
            bits.append((ord(ch) >> i) & 1)
    return bits


def bits_to_text(bits):
    """Convert list of bits back to ASCII text."""
    chars = []
    for i in range(0, len(bits), 8):
        byte = 0
        for b in bits[i:i+8]:
            byte = (byte << 1) | b
        chars.append(chr(byte))
    return ''.join(chars)


# Demonstrate with the textbook example (binary)
T = [0, 1, 1, 0, 1, 1]
K = [1, 1, 1, 0, 1, 0]
E = otp_encrypt(T, K)
D = otp_decrypt(E, K)

print("Textbook Example 9.1.1:")
print(f"  Original message T: {T}")
print(f"  Encryption key K:  {K}")
print(f"  Encrypted E=T^K:   {E}")
print(f"  Decrypted E^K:     {D}")
assert D == T

# Now with actual text
message = "SECRET"
msg_bits = text_to_bits(message)
key = generate_key(len(msg_bits))
cipher_bits = otp_encrypt(msg_bits, key)
recovered = bits_to_text(otp_decrypt(cipher_bits, key))

print(f"\nText example:")
print(f"  Plaintext:  {message}")
print(f"  Recovered:  {recovered}")
assert recovered == message

Textbook Example 9.1.1:
  Original message T: [0, 1, 1, 0, 1, 1]
  Encryption key K:  [1, 1, 1, 0, 1, 0]
  Encrypted E=T^K:   [1, 0, 0, 0, 0, 1]
  Decrypted E^K:     [0, 1, 1, 0, 1, 1]

Text example:
  Plaintext:  SECRET
  Recovered:  SECRET


#### Exercise 9.1.4: Why the key must be used only once

*Suppose Alice generates only one pad key K and uses it to encrypt two
messages $T_1$ and $T_2$ of the same length. Show that by intercepting
$E_1$ and $E_2$, Eve can get $T_1 \oplus T_2$.*

In [5]:
# Demonstrating the danger of key reuse
T1 = text_to_bits("ATTACK")
T2 = text_to_bits("DEFEND")
K = generate_key(len(T1))

E1 = otp_encrypt(T1, K)
E2 = otp_encrypt(T2, K)

# Eve XORs the two ciphertexts: E1 ^ E2 = (T1^K) ^ (T2^K) = T1 ^ T2
eve_xor = [e1 ^ e2 for e1, e2 in zip(E1, E2)]
actual_xor = [t1 ^ t2 for t1, t2 in zip(T1, T2)]

print("Eve computes E1 XOR E2:", eve_xor[:16], "...")
print("Actual T1 XOR T2:     ", actual_xor[:16], "...")
print(f"Match: {eve_xor == actual_xor}")
print("\nThe keys cancel out. Eve now has the XOR of the two plaintexts,")
print("which leaks significant information about both messages.")

Eve computes E1 XOR E2: [0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1] ...
Actual T1 XOR T2:      [0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1] ...
Match: True

The keys cancel out. Eve now has the XOR of the two plaintexts,
which leaks significant information about both messages.


### Private Key vs. Public Key Cryptography

In the One-Time-Pad, knowing $K_E$ immediately gives you $K_D$ (they
are identical). A protocol where both keys must stay secret is called a
**private-key** protocol.

In the 1970s, Rivest, Shamir, and Adleman introduced **RSA**, the first
widely-used **public-key** protocol. Bob generates a key pair $(K_E, K_D)$
where computing $K_D$ from $K_E$ is computationally infeasible. He publishes
$K_E$ for anyone to use. Only he holds $K_D$.

The encryption function $F_E(-) = ENC(-, K_E)$ is a **trapdoor function**:
easy to compute, hard to invert without extra information. Think of a
trapdoor in a castle floor: you fall through easily, but climbing back
up requires a ladder that only the owner has.

Public-key cryptography solves the key distribution problem but relies
on unproven computational hardness assumptions. Quantum computing
(specifically Shor's algorithm) threatens RSA by efficiently factoring
large numbers. The practical response: use public-key methods only to
distribute a shared secret key $K_E$, then switch to a faster private-key
scheme for the actual messages.

Beyond secrecy, cryptographers also care about **intrusion detection**
(is Eve eavesdropping?) and **authentication** (is this really Alice?).
Quantum protocols address the first of these directly.

#### Exercise 9.1.1

*Does Equation (9.3) imply that $ENC(-, K_E)$ and $DEC(-, K_D)$ are
a pair of bijective functions that are inverse to each other?*

**Answer:** No. Equation (9.3) tells us that $DEC(ENC(T, K_E), K_D) = T$
for every $T$. This means $ENC(-, K_E)$ is injective (one-to-one) and
$DEC(-, K_D)$ is surjective (onto). They need not be bijective inverses
in general, because $ENC$ might not hit every possible ciphertext.

#### Exercise 9.1.3

*Find a friend and flip a coin to get an encryption key K. Then use
it to send a message. See if it works.*

This is a hands-on exercise. The code above provides the tools.

#### Exercise 9.1.5

*Suppose Alice and Bob communicate using a public-key protocol. Alice
has a pair of keys (one public, one private), and so does Bob. Devise
a way in which Alice and Bob can communicate simultaneously while
authenticating their messages.*

**Hint:** Think of encoding one message "inside" another. Alice encrypts
with Bob's public key (for secrecy), then signs with her private key
(for authentication). Bob verifies the signature with Alice's public key,
then decrypts with his private key.

## 9.2 Quantum Key Exchange I: The BB84 Protocol

The One-Time-Pad is provably secure, but it has a critical weakness:
Alice and Bob must somehow share a random key $K$ without Eve
intercepting it. In the 1980s, Charles Bennett and Gilles Brassard
solved this with the first **quantum key exchange (QKE)** protocol.

### Why quantum helps

Classical bits can be copied and read without leaving a trace. Qubits
are different:

1. The no-cloning theorem prevents Eve from copying the qubit stream
2. The act of measuring a qubit disturbs it

These seem like limitations, but they become Alice and Bob's greatest
assets. Every time Eve measures a qubit, she disturbs it, and that
disturbance is detectable.

### Two bases

BB84 uses two orthogonal bases (the + and X bases):

$$+ = \{|\!\rightarrow\rangle, |\!\uparrow\rangle\} = \{[1,0]^T, [0,1]^T\} \tag{9.11}$$

$$\mathsf{X} = \{|\!\nearrow\rangle, |\!\searrow\rangle\} = \left\{\frac{1}{\sqrt{2}}[-1,1]^T, \frac{1}{\sqrt{2}}[1,1]^T\right\} \tag{9.12}$$

The + basis is the standard (rectilinear) basis; the X basis is the
diagonal (Hadamard) basis. In these vocabularies:

| State/Basis | + | X |
|---|---|---|
| $|0\rangle$ | $|\!\rightarrow\rangle$ | $|\!\nearrow\rangle$ |
| $|1\rangle$ | $|\!\uparrow\rangle$ | $|\!\searrow\rangle$ |

When bases mismatch (Alice sends in +, Bob measures in X, or vice versa),
the result is a fair coin flip: 50-50 chance of reading 0 or 1.

#### Exercise 9.2.1

*Work out what $|\!\leftarrow\rangle$, $|\!\downarrow\rangle$, $|\!\swarrow\rangle$, and $|\!\searrow\rangle$ would be in terms of the two bases.*

**Answer** (from Appendix B):
- $|\!\leftarrow\rangle$ with respect to + will be $-1|\!\rightarrow\rangle$
- $|\!\downarrow\rangle$ with respect to + will be $-1|\!\uparrow\rangle$
- $|\!\swarrow\rangle$ with respect to + will be $\frac{1}{\sqrt{2}}|\!\uparrow\rangle - \frac{1}{\sqrt{2}}|\!\rightarrow\rangle$
- $|\!\searrow\rangle$ with respect to + will be $-\frac{1}{\sqrt{2}}|\!\uparrow\rangle + \frac{1}{\sqrt{2}}|\!\rightarrow\rangle$
- $|\!\leftarrow\rangle$ with respect to X will be $-\frac{1}{\sqrt{2}}|\!\nearrow\rangle + \frac{1}{\sqrt{2}}|\!\searrow\rangle$
- $|\!\downarrow\rangle$ with respect to X will be $-\frac{1}{\sqrt{2}}|\!\nearrow\rangle - \frac{1}{\sqrt{2}}|\!\searrow\rangle$
- $|\!\swarrow\rangle$ with respect to X will be $-\frac{1}{\sqrt{2}}|\!\nearrow\rangle$
- $|\!\searrow\rangle$ with respect to X will be $-\frac{1}{\sqrt{2}}|\!\searrow\rangle$

### The BB84 Protocol

**Step 1.** Alice flips a coin $n$ times to choose random bits. She flips
again $n$ times to choose random bases (+  or X). She sends each bit
encoded in its corresponding basis.

**Step 2.** Bob doesn't know Alice's bases, so he also picks random bases
and measures each qubit. When their bases match (about half the time),
Bob's result equals Alice's bit. When they differ, Bob gets a random
result.

**Step 3.** Alice and Bob publicly compare which *bases* they used (not the
bit values). They discard every bit where bases disagreed. The remaining
~$n/2$ bits should match exactly.

**Step 4.** To check for Eve, Bob randomly selects half of the remaining bits
and publicly compares them with Alice. If more than a tiny percentage
disagree (beyond noise), Eve was listening. They throw away the entire
sequence. If they agree, the unrevealed bits become the secret key,
roughly $n/4$ bits long.

#### Programming Drill 9.2.1

*Write functions that mimic Alice, Bob, and their interactions. Alice
generates two random bit strings (BitSent and SendingBasis). Bob
generates a random ReceivingBasis. An "all-knowing" function Knuth
creates BitReceived according to:*

$$BitReceived[i] = \begin{cases} BitSent[i] & \text{if } SendingBasis[i] = ReceivingBasis[i] \\ random\{0,1\} & \text{otherwise} \end{cases} \tag{9.17}$$

In [6]:
def alice_prepare(n):
    """Alice generates random bits and random bases."""
    bits = [random.randint(0, 1) for _ in range(n)]
    bases = [random.choice(['+', 'X']) for _ in range(n)]
    return bits, bases


def bob_measure(n):
    """Bob picks random measurement bases."""
    return [random.choice(['+', 'X']) for _ in range(n)]


def knuth(alice_bits, alice_bases, bob_bases):
    """Simulate the quantum channel (no eavesdropper).
    
    If bases match, Bob gets Alice's bit exactly.
    If bases differ, Bob gets a random bit.
    Returns (bob_bits, agreement_rate).
    """
    n = len(alice_bits)
    bob_bits = []
    for i in range(n):
        if alice_bases[i] == bob_bases[i]:
            bob_bits.append(alice_bits[i])
        else:
            bob_bits.append(random.randint(0, 1))
    return bob_bits


# Run BB84 without Eve
random.seed(42)
n = 100
alice_bits, alice_bases = alice_prepare(n)
bob_bases = bob_measure(n)
bob_bits = knuth(alice_bits, alice_bases, bob_bases)

# Step 3: Compare bases publicly, keep matching positions
matching = [i for i in range(n) if alice_bases[i] == bob_bases[i]]
print(f"Bits sent: {n}")
print(f"Bases matched: {len(matching)} (expected ~{n//2})")

# Extract shared bits from matching positions
alice_shared = [alice_bits[i] for i in matching]
bob_shared = [bob_bits[i] for i in matching]
agreement = sum(a == b for a, b in zip(alice_shared, bob_shared))
print(f"Agreement on shared bits: {agreement}/{len(matching)} = {agreement/len(matching):.1%}")
print(f"(Without Eve, this should be 100%)")

Bits sent: 100
Bases matched: 58 (expected ~50)
Agreement on shared bits: 58/58 = 100.0%
(Without Eve, this should be 100%)


#### Programming Drill 9.2.2

*Write a function Knuth2 that accepts all three bit strings and creates
AgreedBits, a substring of both BitSent and BitReceived.*

In [7]:
def knuth2(alice_bits, alice_bases, bob_bases, bob_bits):
    """Extract the agreed-upon key bits where bases matched."""
    agreed = []
    for i in range(len(alice_bits)):
        if alice_bases[i] == bob_bases[i]:
            agreed.append(alice_bits[i])
    return agreed


agreed_key = knuth2(alice_bits, alice_bases, bob_bases, bob_bits)
print(f"Agreed key length: {len(agreed_key)}")
print(f"First 20 key bits: {agreed_key[:20]}")

Agreed key length: 58
First 20 key bits: [0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0]


### Eavesdropper Detection

If Eve intercepts and measures each qubit before forwarding it to Bob,
she must guess the basis. Half the time she guesses wrong, collapsing
the qubit to one of *her* basis states. When Bob then measures with
Alice's original basis, he gets a random result for those qubits Eve
measured in the wrong basis.

The error rate: Eve guesses the wrong basis 50% of the time. In those
cases, Bob gets the wrong bit 50% of the time. So overall,
25% of the bits where Alice and Bob's bases matched will disagree.

In [8]:
def eve_intercept(alice_bits, alice_bases):
    """Eve measures each qubit in a random basis, then forwards her result."""
    n = len(alice_bits)
    eve_bases = [random.choice(['+', 'X']) for _ in range(n)]
    eve_bits = []
    for i in range(n):
        if eve_bases[i] == alice_bases[i]:
            eve_bits.append(alice_bits[i])
        else:
            eve_bits.append(random.randint(0, 1))
    return eve_bits, eve_bases


def knuth_with_eve(eve_bits, eve_bases, bob_bases):
    """Bob receives qubits that Eve has already measured and forwarded.
    
    The qubits are now in Eve's basis states, not Alice's original.
    """
    bob_bits = []
    for i in range(len(eve_bits)):
        if eve_bases[i] == bob_bases[i]:
            bob_bits.append(eve_bits[i])
        else:
            bob_bits.append(random.randint(0, 1))
    return bob_bits


# Full BB84 simulation with Eve
random.seed(123)
n = 1000
alice_bits, alice_bases = alice_prepare(n)
eve_bits, eve_bases = eve_intercept(alice_bits, alice_bases)
bob_bases = bob_measure(n)
bob_bits = knuth_with_eve(eve_bits, eve_bases, bob_bases)

# Compare only where Alice and Bob used the same basis
matching = [i for i in range(n) if alice_bases[i] == bob_bases[i]]
errors = sum(alice_bits[i] != bob_bits[i] for i in matching)
error_rate = errors / len(matching)

print(f"BB84 with eavesdropper (n={n}):")
print(f"  Matching bases: {len(matching)}")
print(f"  Errors in shared bits: {errors}")
print(f"  Error rate: {error_rate:.1%} (expected ~25%)")
print(f"\n  Alice and Bob detect Eve: error rate >> 0")

BB84 with eavesdropper (n=1000):
  Matching bases: 512
  Errors in shared bits: 118
  Error rate: 23.0% (expected ~25%)

  Alice and Bob detect Eve: error rate >> 0


With no Eve, the error rate on matching-basis bits is 0%. With Eve
intercepting every qubit, the error rate rises to approximately 25%.
Alice and Bob can detect this by publicly comparing a random subset
of their shared bits.

#### Exercise 9.2.2

*Give an estimate of how frequently Bob's bit will agree with Alice's
if Eve is constantly eavesdropping.*

When Eve intercepts, she guesses the correct basis with probability 1/2.
When she guesses right, she forwards Alice's bit perfectly. When she guesses
wrong, the qubit collapses to one of Eve's basis states. Bob (measuring in
Alice's basis) then has a 50-50 chance.

Overall agreement (on matching-basis bits): $\frac{1}{2}(1) + \frac{1}{2}(\frac{1}{2}) = \frac{3}{4}$.

So Bob's bit agrees with Alice's about **75% of the time** (25% error rate).

## 9.3 Quantum Key Exchange II: The B92 Protocol

BB84 uses two complete orthogonal bases. It turns out that one
*nonorthogonal* basis suffices. The B92 protocol (Bennett, 1992) uses
just two states:

$$\{|\!\rightarrow\rangle, |\!\nearrow\rangle\} = \left\{[1,0]^T, \frac{1}{\sqrt{2}}[1,1]^T\right\} \tag{9.19}$$

These two vectors are *not* orthogonal. No single measurement can
reliably distinguish between them. This is the key insight: the
impossibility of perfect discrimination provides security.

#### Exercise 9.3.1

*Verify that these two vectors are, in fact, not orthogonal.*

In [9]:
# Exercise 9.3.1: verify nonorthogonality
v0 = np.array([1, 0])           # |right arrow>
v1 = np.array([1, 1]) / sqrt(2) # |diagonal arrow>

inner_product = np.dot(v0, v1)
print(f"Inner product: {inner_product:.4f}")
print(f"Orthogonal? {np.isclose(inner_product, 0)}")
print(f"\nThey are NOT orthogonal (inner product = 1/sqrt(2) ≈ 0.7071).")

Inner product: 0.7071
Orthogonal? False

They are NOT orthogonal (inner product = 1/sqrt(2) ≈ 0.7071).


### B92 Protocol Steps

Alice maps: $0 \to |\!\rightarrow\rangle$ and $1 \to |\!\nearrow\rangle$.

**Step 1.** Alice flips a coin $n$ times and sends each bit in the
appropriate state (all using the single nonorthogonal basis).

**Step 2.** For each qubit, Bob measures in either the + or X basis
(chosen by coin flip). Four scenarios:

- Bob uses + and sees $|\!\uparrow\rangle$: Alice must have sent
  $|\!\nearrow\rangle = |1\rangle$. (If she sent $|\!\rightarrow\rangle$,
  Bob could never see $|\!\uparrow\rangle$.) Bob records 1.
- Bob uses + and sees $|\!\rightarrow\rangle$: ambiguous. Discard.
- Bob uses X and sees $|\!\searrow\rangle$: Alice must have sent
  $|\!\rightarrow\rangle = |0\rangle$. Bob records 0.
- Bob uses X and sees $|\!\nearrow\rangle$: ambiguous. Discard.

For half the results Bob has certainty; for the other half he discards.

**Step 3.** Bob publicly tells Alice which bits were uncertain. They both
discard those. The remaining bits form the shared secret. They can
sacrifice half for eavesdropper detection, as in BB84.

#### Programming Drill 9.3.1

*Write three functions named Alice92, Bob92, and Knuth92 that perform
the B92 protocol.*

In [10]:
def alice92(n):
    """Alice generates n random bits. Sends |right> for 0, |diag> for 1."""
    return [random.randint(0, 1) for _ in range(n)]


def bob92_bases(n):
    """Bob picks random measurement bases."""
    return [random.choice(['+', 'X']) for _ in range(n)]


def knuth92(alice_bits, bob_bases):
    """Simulate B92 quantum channel.
    
    Returns (bob_bits, certain) where certain[i] indicates
    whether Bob is certain about bit i.
    """
    n = len(alice_bits)
    bob_bits = []
    certain = []
    
    for i in range(n):
        if alice_bits[i] == 0:  # Alice sent |right> = [1,0]
            if bob_bases[i] == '+':
                # Measuring |right> in + basis: always get |right> = 0
                bob_bits.append(0)
                certain.append(False)  # Could be either, ambiguous
            else:
                # Measuring |right> in X basis: 50-50
                if random.random() < 0.5:
                    bob_bits.append(0)  # got |diag_up>: ambiguous
                    certain.append(False)
                else:
                    bob_bits.append(0)  # got |diag_down>: certain it was |right>=0
                    certain.append(True)
        else:  # alice_bits[i] == 1, Alice sent |diag> = 1/sqrt(2)[1,1]
            if bob_bases[i] == 'X':
                # Measuring |diag> in X basis: always get |diag> = 1
                bob_bits.append(1)
                certain.append(False)  # ambiguous
            else:
                # Measuring |diag> in + basis: 50-50
                if random.random() < 0.5:
                    bob_bits.append(1)  # got |right>: ambiguous
                    certain.append(False)
                else:
                    bob_bits.append(1)  # got |up>: certain it was |diag>=1
                    certain.append(True)
    
    return bob_bits, certain


# Run B92
random.seed(99)
n = 200
alice_bits = alice92(n)
bob_bases = bob92_bases(n)
bob_bits, certain = knuth92(alice_bits, bob_bases)

# Keep only certain bits
alice_key = [alice_bits[i] for i in range(n) if certain[i]]
bob_key = [bob_bits[i] for i in range(n) if certain[i]]

agreement = sum(a == b for a, b in zip(alice_key, bob_key))
print(f"B92 Protocol (n={n}):")
print(f"  Certain bits: {len(alice_key)} (expected ~{n//4})")
print(f"  Agreement: {agreement}/{len(alice_key)} = {agreement/len(alice_key):.1%}")
print(f"  First 15 key bits (Alice): {alice_key[:15]}")
print(f"  First 15 key bits (Bob):   {bob_key[:15]}")

B92 Protocol (n=200):
  Certain bits: 43 (expected ~50)
  Agreement: 43/43 = 100.0%
  First 15 key bits (Alice): [1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0]
  First 15 key bits (Bob):   [1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0]


## 9.4 Quantum Key Exchange III: The EPR Protocol

In 1991, Artur Ekert proposed a completely different approach: use
entanglement for key distribution.

Place two qubits in the entangled state:

$$\frac{|00\rangle + |11\rangle}{\sqrt{2}} \tag{9.22}$$

When one qubit is measured, both collapse to the same value. Give
one qubit to Alice, the other to Bob. They both measure, and they
now share a random bit that nobody else knows.

### Simplified EPR Protocol

**Step 1.** Alice and Bob each receive one qubit from a sequence of
entangled pairs.

**Step 2.** Each independently chooses a random basis (+ or X) and
measures their qubit.

**Step 3.** They publicly compare bases and keep only the bits where
they used the same basis. (When bases match, entanglement guarantees
their results agree.)

**Step 4.** To detect Eve (who might have intercepted and disentangled
the pairs), they sacrifice half their shared bits for comparison.
Entangled particles violate **Bell's inequality**; classical imposters do
not. If the test subsequence violates Bell's inequality, the particles
were genuinely entangled. If it satisfies the inequality, Eve may have
tampered with the pairs, and the sequence should be discarded.

Ekert's original version is even more sophisticated: Alice and Bob
measure in three different bases and use Bell's inequality directly to
verify entanglement.

In [11]:
def epr_protocol(n):
    """Simulate the simplified EPR key distribution protocol."""
    # Generate entangled pairs: when measured in same basis, results match
    shared_values = [random.randint(0, 1) for _ in range(n)]
    
    alice_bases = [random.choice(['+', 'X']) for _ in range(n)]
    bob_bases = [random.choice(['+', 'X']) for _ in range(n)]
    
    # Alice and Bob measure
    alice_bits = []
    bob_bits = []
    for i in range(n):
        if alice_bases[i] == bob_bases[i]:
            # Same basis: entanglement guarantees same result
            alice_bits.append(shared_values[i])
            bob_bits.append(shared_values[i])
        else:
            # Different basis: random, uncorrelated results
            alice_bits.append(random.randint(0, 1))
            bob_bits.append(random.randint(0, 1))
    
    # Step 3: publicly compare bases
    matching = [i for i in range(n) if alice_bases[i] == bob_bases[i]]
    alice_key = [alice_bits[i] for i in matching]
    bob_key = [bob_bits[i] for i in matching]
    
    return alice_key, bob_key


random.seed(77)
alice_key, bob_key = epr_protocol(1000)
agreement = sum(a == b for a, b in zip(alice_key, bob_key))
print(f"EPR Protocol (n=1000):")
print(f"  Shared key length: {len(alice_key)} (expected ~500)")
print(f"  Agreement: {agreement}/{len(alice_key)} = {agreement/len(alice_key):.1%}")
print(f"  (Should be 100% without Eve)")

EPR Protocol (n=1000):
  Shared key length: 474 (expected ~500)
  Agreement: 474/474 = 100.0%
  (Should be 100% without Eve)


## 9.5 Quantum Teleportation

**Quantum teleportation** is the process by which the state of an
arbitrary qubit is transferred from one location to another.

This is not science fiction. It has been performed in the laboratory.
The protocol does not move matter or transmit information faster than
light. It transfers a quantum *state* using a shared entangled pair and
two classical bits.

The no-cloning theorem means that teleportation necessarily *destroys*
the original. "Move is possible. Copy is impossible."

### The Bell Basis

Working with entanglement requires switching between the canonical basis
and a noncanonical one. For two qubits, the canonical basis is:

$$\{|0_A 0_B\rangle, |0_A 1_B\rangle, |1_A 0_B\rangle, |1_A 1_B\rangle\} \tag{9.28}$$

The **Bell basis**, named after John Bell, consists of four entangled states:

$$|\Psi^+\rangle = \frac{|0_A 1_B\rangle + |1_A 0_B\rangle}{\sqrt{2}} \tag{9.29}$$

$$|\Psi^-\rangle = \frac{|0_A 1_B\rangle - |1_A 0_B\rangle}{\sqrt{2}} \tag{9.30}$$

$$|\Phi^+\rangle = \frac{|0_A 0_B\rangle + |1_A 1_B\rangle}{\sqrt{2}} \tag{9.31}$$

$$|\Phi^-\rangle = \frac{|0_A 0_B\rangle - |1_A 1_B\rangle}{\sqrt{2}} \tag{9.32}$$

Every state in this basis is entangled. These four vectors form a complete
basis for $\mathbb{C}^2 \otimes \mathbb{C}^2$. Every canonical basis vector
can be expressed in terms of Bell states and vice versa:

$$|0_A 0_B\rangle = \frac{1}{\sqrt{2}}(|\Phi^+\rangle + |\Phi^-\rangle) \tag{9.33}$$
$$|1_A 1_B\rangle = \frac{1}{\sqrt{2}}(|\Phi^+\rangle - |\Phi^-\rangle) \tag{9.34}$$
$$|0_A 1_B\rangle = \frac{1}{\sqrt{2}}(|\Psi^+\rangle + |\Psi^-\rangle) \tag{9.35}$$
$$|1_A 0_B\rangle = \frac{1}{\sqrt{2}}(|\Psi^+\rangle - |\Psi^-\rangle) \tag{9.36}$$

In [12]:
# Define the Bell basis states
ket0 = np.array([1, 0], dtype=complex)
ket1 = np.array([0, 1], dtype=complex)

# Canonical two-qubit basis
ket00 = np.kron(ket0, ket0)
ket01 = np.kron(ket0, ket1)
ket10 = np.kron(ket1, ket0)
ket11 = np.kron(ket1, ket1)

# Bell states
psi_plus  = (ket01 + ket10) / sqrt(2)
psi_minus = (ket01 - ket10) / sqrt(2)
phi_plus  = (ket00 + ket11) / sqrt(2)
phi_minus = (ket00 - ket11) / sqrt(2)

print("Bell basis states:")
for name, state in [("Psi+", psi_plus), ("Psi-", psi_minus),
                     ("Phi+", phi_plus), ("Phi-", phi_minus)]:
    print(f"  |{name}> = {np.round(state, 4)}")

# Verify orthonormality
bell_states = [psi_plus, psi_minus, phi_plus, phi_minus]
print("\nOrthonormality check:")
for i, si in enumerate(bell_states):
    for j, sj in enumerate(bell_states):
        ip = np.dot(si.conj(), sj)
        if not np.isclose(ip, 0):
            print(f"  <Bell[{i}]|Bell[{j}]> = {ip.real:.0f}")

Bell basis states:
  |Psi+> = [0.    +0.j 0.7071+0.j 0.7071+0.j 0.    +0.j]
  |Psi-> = [ 0.    +0.j  0.7071+0.j -0.7071+0.j  0.    +0.j]
  |Phi+> = [0.7071+0.j 0.    +0.j 0.    +0.j 0.7071+0.j]
  |Phi-> = [ 0.7071+0.j  0.    +0.j  0.    +0.j -0.7071+0.j]

Orthonormality check:
  <Bell[0]|Bell[0]> = 1
  <Bell[1]|Bell[1]> = 1
  <Bell[2]|Bell[2]> = 1
  <Bell[3]|Bell[3]> = 1


### Building Bell States from a Circuit

The Hadamard gate $H$ and CNOT gate together produce Bell states:

$$|00\rangle \mapsto |\Phi^+\rangle, \quad |01\rangle \mapsto |\Psi^+\rangle, \quad |10\rangle \mapsto |\Phi^-\rangle, \quad |11\rangle \mapsto |\Psi^-\rangle \tag{9.39}$$

In [13]:
# Hadamard
H = np.array([[1, 1], [1, -1]], dtype=complex) / sqrt(2)

# CNOT
CNOT = np.array([[1,0,0,0],
                  [0,1,0,0],
                  [0,0,0,1],
                  [0,0,1,0]], dtype=complex)

# H on first qubit, identity on second
H_I = np.kron(H, np.eye(2))

# Bell circuit: CNOT @ (H tensor I)
bell_circuit = CNOT @ H_I

# Verify the mappings from Eq (9.39)
print("Bell circuit mappings:")
inputs = [("00", ket00), ("01", ket01), ("10", ket10), ("11", ket11)]
expected = [("Phi+", phi_plus), ("Psi+", psi_plus),
            ("Phi-", phi_minus), ("Psi-", psi_minus)]

for (in_name, in_state), (out_name, out_state) in zip(inputs, expected):
    result = bell_circuit @ in_state
    match = np.allclose(result, out_state)
    print(f"  |{in_name}> -> |{out_name}>: {match}")

Bell circuit mappings:
  |00> -> |Phi+>: True
  |01> -> |Psi+>: True
  |10> -> |Phi->: True
  |11> -> |Psi->: True


### The Teleportation Protocol

Alice has a qubit $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ she wants
to teleport to Bob. Neither knows $\alpha$ or $\beta$.

**Step 1.** Two entangled qubits are formed as $|\Phi^+\rangle$. One goes
to Alice (qubit A), one to Bob (qubit B). The full system of three
qubits is:

$$|\varphi_2\rangle = |\psi\rangle \otimes |\Phi^+\rangle = (\alpha|0\rangle + \beta|1\rangle) \otimes \frac{|0_A 0_B\rangle + |1_A 1_B\rangle}{\sqrt{2}} \tag{9.43}$$

**Step 2.** Alice lets $|\psi\rangle$ interact with her entangled qubit A through
a CNOT gate (with $|\psi\rangle$ as control) followed by a Hadamard on
$|\psi\rangle$.

After this interaction, the three-qubit state becomes:

$$|\varphi_4\rangle = \frac{1}{2}\big(|00\rangle(\alpha|0\rangle + \beta|1\rangle) + |01\rangle(\beta|0\rangle + \alpha|1\rangle)$$
$$+ |10\rangle(\alpha|0\rangle - \beta|1\rangle) + |11\rangle(-\beta|0\rangle + \alpha|1\rangle)\big) \tag{9.46}$$

**Step 3.** Alice measures her two qubits, obtaining one of four outcomes.
The system collapses and Bob's qubit is now in a known (but possibly
rotated) version of $|\psi\rangle$.

**Step 4.** Alice sends her two classical bits to Bob, who applies a
correction matrix:

| Bits received | Matrix to apply |
|---|---|
| 00 | $I = \begin{bmatrix}1&0\\0&1\end{bmatrix}$ |
| 01 | $X = \begin{bmatrix}0&1\\1&0\end{bmatrix}$ |
| 10 | $Z = \begin{bmatrix}1&0\\0&-1\end{bmatrix}$ |
| 11 | $iY = \begin{bmatrix}0&-1\\1&0\end{bmatrix}$ | (9.49)

After applying the matrix, Bob's qubit is exactly $|\psi\rangle$.

In [14]:
def teleport(alpha, beta):
    """Simulate quantum teleportation of state alpha|0> + beta|1>.
    
    Returns (classical_bits, bob_final_state).
    """
    # The state to teleport
    psi = np.array([alpha, beta], dtype=complex)
    assert np.isclose(norm(psi), 1.0), "State must be normalized"
    
    # Step 1: Create entangled pair |Phi+> = (|00>+|11>)/sqrt(2)
    # Full 3-qubit state: |psi> tensor |Phi+>
    phi_plus_2q = (ket00 + ket11) / sqrt(2)
    state = np.kron(psi, phi_plus_2q)  # 8-dimensional
    
    # Step 2: Alice applies CNOT (psi=control, A=target) then H on psi
    # CNOT on qubits 0,1 (tensor I on qubit 2)
    CNOT_01 = np.kron(CNOT, np.eye(2))
    state = CNOT_01 @ state
    
    # H on qubit 0 (tensor I on qubits 1,2)
    H_I_I = np.kron(np.kron(H, np.eye(2)), np.eye(2))
    state = H_I_I @ state
    
    # Step 3: Alice measures qubits 0 and 1
    # Probability of each 2-bit outcome
    probs = {}
    for bits in range(4):
        # Project onto |bits> for first two qubits
        proj_state = state[bits*2:(bits+1)*2]
        probs[bits] = np.real(np.dot(proj_state.conj(), proj_state))
    
    # Randomly choose outcome according to probabilities
    outcomes = list(probs.keys())
    weights = [probs[o] for o in outcomes]
    measured = random.choices(outcomes, weights=weights, k=1)[0]
    
    # Bob's qubit after measurement
    bob_state = state[measured*2:(measured+1)*2]
    bob_state = bob_state / norm(bob_state)  # renormalize
    
    # Step 4: Bob applies correction based on Alice's bits
    I2 = np.eye(2, dtype=complex)
    X_gate = np.array([[0, 1], [1, 0]], dtype=complex)
    Z_gate = np.array([[1, 0], [0, -1]], dtype=complex)
    iY_gate = np.array([[0, -1], [1, 0]], dtype=complex)
    
    corrections = {0: I2, 1: X_gate, 2: Z_gate, 3: iY_gate}
    bob_final = corrections[measured] @ bob_state
    
    classical_bits = f"{measured:02b}"
    return classical_bits, bob_final


# Test with several states
random.seed(42)
test_states = [
    ("alpha=1, beta=0", 1+0j, 0+0j),
    ("alpha=0, beta=1", 0+0j, 1+0j),
    ("equal superposition", 1/sqrt(2)+0j, 1/sqrt(2)+0j),
    ("arbitrary state", 0.6+0j, 0.8+0j),
    ("complex phase", (1+1j)/2, (1-1j)/2),
]

print("Teleportation results:")
print("="*65)
for name, a, b in test_states:
    original = np.array([a, b], dtype=complex)
    bits, result = teleport(a, b)
    # States equal up to global phase
    phase = result[0] / original[0] if abs(original[0]) > 1e-10 else result[1] / original[1]
    adjusted = result / phase if abs(phase) > 1e-10 else result
    match = np.allclose(adjusted, original)
    print(f"  {name}")
    print(f"    Original:    [{a:.4f}, {b:.4f}]")
    print(f"    Teleported:  [{result[0]:.4f}, {result[1]:.4f}]")
    print(f"    Classical bits: {bits}")
    print(f"    Match (up to global phase): {match}")
    print()

Teleportation results:
  alpha=1, beta=0
    Original:    [1.0000+0.0000j, 0.0000+0.0000j]
    Teleported:  [1.0000+0.0000j, 0.0000+0.0000j]
    Classical bits: 10
    Match (up to global phase): True

  alpha=0, beta=1
    Original:    [0.0000+0.0000j, 1.0000+0.0000j]
    Teleported:  [0.0000+0.0000j, 1.0000+0.0000j]
    Classical bits: 00
    Match (up to global phase): True

  equal superposition
    Original:    [0.7071+0.0000j, 0.7071+0.0000j]
    Teleported:  [0.7071+0.0000j, 0.7071+0.0000j]
    Classical bits: 01
    Match (up to global phase): True

  arbitrary state
    Original:    [0.6000+0.0000j, 0.8000+0.0000j]
    Teleported:  [0.6000+0.0000j, 0.8000+0.0000j]
    Classical bits: 00
    Match (up to global phase): True

  complex phase
    Original:    [0.5000+0.5000j, 0.5000-0.5000j]
    Teleported:  [0.5000+0.5000j, 0.5000-0.5000j]
    Classical bits: 10
    Match (up to global phase): True



### Key Points About Teleportation

- Alice no longer has $|\psi\rangle$ after the protocol. She has two classical bits.
  The original state is destroyed (no cloning).
- To teleport one qubit, Alice must send two classical bits. Without them,
  Bob has no way to know which correction to apply. These bits travel at
  the speed of light (or slower). Entanglement does *not* allow faster-than-light
  communication. Einstein's relativity is safe.
- $\alpha$ and $\beta$ can be arbitrary complex numbers satisfying
  $|\alpha|^2 + |\beta|^2 = 1$. A potentially infinite amount of information
  has crossed from Alice to Bob, but it arrives as a *qubit*, not as classical
  data. The moment Bob measures it, it collapses to a single bit.
- From the standpoint of physics, two particles in exactly the same quantum
  state are indistinguishable. If you are configured like Captain Kirk down
  to the last quantum detail, you *are* Captain Kirk.

#### Exercise 9.5.1

*What about Eve? She can certainly tap into the classical channel and
snatch the two bits. But will it be useful to her?*

No. The two classical bits (00, 01, 10, or 11) tell Bob which correction
to apply, but without access to the entangled qubit itself, the bits are
meaningless to Eve. They carry no information about $\alpha$ or $\beta$;
each outcome is equally likely regardless of the state being teleported.

#### Exercise 9.5.2

*There was nothing special about using $|\Phi^+\rangle$ to entangle Alice's
qubits with $|\psi\rangle$. She could have used any of the other three Bell
vectors. Work out the result if she had used $|\Phi^-\rangle$.*

Let us verify computationally that teleportation works with each Bell state.

In [15]:
def teleport_with_bell_state(alpha, beta, bell_state):
    """Teleportation using an arbitrary Bell state for the entangled pair."""
    psi = np.array([alpha, beta], dtype=complex)
    
    # 3-qubit state: |psi> tensor bell_state
    state = np.kron(psi, bell_state)
    
    # Alice's operations: CNOT on qubits 0,1 then H on qubit 0
    CNOT_01 = np.kron(CNOT, np.eye(2))
    H_I_I = np.kron(np.kron(H, np.eye(2)), np.eye(2))
    state = H_I_I @ (CNOT_01 @ state)
    
    # Check all four measurement outcomes
    print(f"  Bob's state for each measurement outcome:")
    for bits in range(4):
        bob = state[bits*2:(bits+1)*2]
        if norm(bob) > 1e-10:
            bob_normed = bob / norm(bob)
            print(f"    |{bits:02b}>: [{bob_normed[0]:.4f}, {bob_normed[1]:.4f}]")


alpha, beta = 0.6+0j, 0.8+0j
print(f"Teleporting [{alpha}, {beta}]")
print(f"\nUsing |Phi+>:")
teleport_with_bell_state(alpha, beta, phi_plus)
print(f"\nUsing |Phi->:")
teleport_with_bell_state(alpha, beta, phi_minus)
print(f"\nUsing |Psi+>:")
teleport_with_bell_state(alpha, beta, psi_plus)
print(f"\nUsing |Psi->:")
teleport_with_bell_state(alpha, beta, psi_minus)
print(f"\nIn each case, Bob can recover [{alpha}, {beta}] with the right correction.")

Teleporting [(0.6+0j), (0.8+0j)]

Using |Phi+>:
  Bob's state for each measurement outcome:
    |00>: [0.6000+0.0000j, 0.8000+0.0000j]
    |01>: [0.8000+0.0000j, 0.6000+0.0000j]
    |10>: [0.6000+0.0000j, -0.8000+0.0000j]
    |11>: [-0.8000+0.0000j, 0.6000+0.0000j]

Using |Phi->:
  Bob's state for each measurement outcome:
    |00>: [0.6000+0.0000j, -0.8000+0.0000j]
    |01>: [0.8000+0.0000j, -0.6000+0.0000j]
    |10>: [0.6000+0.0000j, 0.8000+0.0000j]
    |11>: [-0.8000+0.0000j, -0.6000+0.0000j]

Using |Psi+>:
  Bob's state for each measurement outcome:
    |00>: [0.8000+0.0000j, 0.6000+0.0000j]
    |01>: [0.6000+0.0000j, 0.8000+0.0000j]
    |10>: [-0.8000+0.0000j, 0.6000+0.0000j]
    |11>: [0.6000+0.0000j, -0.8000+0.0000j]

Using |Psi->:
  Bob's state for each measurement outcome:
    |00>: [-0.8000+0.0000j, 0.6000+0.0000j]
    |01>: [-0.6000+0.0000j, 0.8000+0.0000j]
    |10>: [0.8000+0.0000j, 0.6000+0.0000j]
    |11>: [-0.6000+0.0000j, -0.8000+0.0000j]

In each case, Bob can recover 

## Summary

Classical cryptography gives us the tools (ENC/DEC, One-Time-Pad),
but stumbles on key distribution. Quantum mechanics provides three
solutions, each exploiting a different physical principle:

- **BB84**: measurement disturbance in two bases detects eavesdroppers
- **B92**: nonorthogonal states achieve the same with a simpler setup
- **EPR**: entanglement generates shared randomness; Bell's inequality
  certifies its integrity

Quantum teleportation rounds out the picture. A quantum state can be
transferred using shared entanglement and classical communication.
No faster-than-light signaling, no cloning, no violations of relativity.
Just linear algebra applied to a physical substrate that happens to be
quantum mechanical.